# 3. Model configuration and training

Model construction is staged: 
- select a model family and name, 
- configure components or apply a preset, 
- configure a dataset, 
- compile, 
- estimate resources, 
- train. 

Compilation creates the PyTorch module and dataset but marks the model untrained until `fit` succeeds.

In [1]:
import os 
from pathlib import Path

# seting global dir
cwd=Path.cwd()
if cwd.name == "tutorials":
    # os.chdir(cwd.parent.parent) 
    os.chdir(cwd.parent.parent.parent) 
os.getcwd()

'/home/maxi7524/repositories/MSIAutoEncoderWrapper'

In [ ]:
from pathlib import Path
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

wrapper = MSIAutoEncoderWrapper("data/tutorial_workspace")
# more optimal is code below, but here we want to ensure user provided right structure
# wrapper.workspace.set_default_image_path('example')
image_path = Path("data/tutorial_workspace/imgs/example.imzML").resolve()
wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
wrapper.context_manager.set_binner("LinearBinning", str(image_path), bin_step=0.1)
wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner", str(image_path), max_bins=1500, window_size=3
)
wrapper.workspace.set_active_image(str(image_path))

2026-07-19 16:07:15,700 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets'.
2026-07-19 16:07:15,703 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 14 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders'.
2026-07-19 16:07:15,704 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 0 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.schema'.
2026-07-19 16:07:15,706 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 20 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.
2026-07-19 16:07:15,719 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 6 implementation module(s) in package 'msi_autoencoder_wrapper.training.criterions.autoencoder'.
2026-07-19 16

/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession MS:1000563 found with incorrect name "Thermo RAW file". Updating name to "Thermo RAW format".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession MS:1000590 found with incorrect name "contact organization". Updating name to "contact affiliation".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000042 found with incorrect name "max count of pixel x". Updating name to "max count of pixels x".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000043 found with incorrect name "max count of pixel y". Updating name to "max count of pixels y".
  warn(


2026-07-19 16:07:21,327 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:338 | Successfully registered component 'reader' into ledger for image 'example'
2026-07-19 16:07:21,329 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:69 | Active image set via direct filesystem path: example (Location: /home/maxi7524/repositories/MSIAutoEncoderWrapper/data/tutorial_workspace/imgs)
2026-07-19 16:07:21,329 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:323 | Resolving system component 'binner' under image context 'example'
2026-07-19 16:07:21,330 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:108 | Successfully bound active context memory maps for: example
2026-07-19 16:07:21,332 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:338 | Successfully registered component 'binner' into ledger for image 'ex

## Inspect the registries

A family determines the valid component categories, criteria, and runtime interface. Discovery is useful before writing a config and returns constructor metadata when `return_value=True`.

In [3]:
# Here we search through all available models and set autoencoder
wrapper.models_manager.get_available_model_types()
wrapper.models_manager.set_model_type("autoencoder", "tutorial-autoencoder")


 Available Master Model Topologies

[Model Type]: 'autoencoder'
 Description: Symmetric architectural backbone coordinating data transformations across autoencoder blocks.
 Parameters (kwargs):
   - resolved_components: Required



In [4]:
# Here we search through available components for given model (autoencoder) 
wrapper.models_manager.get_available_component_categories()
wrapper.models_manager.get_available_model_presets()
wrapper.models_manager.get_available_criterions()
# Search through dataset is independent
wrapper.models_manager.get_available_datasets()


 Registered Component Categories for 'autoencoder'

[Category]: 'decoder'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None

[Category]: 'encoder'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None

[Category]: 'projector'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None


 Available Configuration Presets for 'autoencoder'

[Preset]: 'GradualReduction'
 Description: Dynamically suggests compatible neural structures based on raw peak widths metrics.

Estimates peak envelope widths to configure matching initial convolutional fields,
iteratively scaling hidden depths until layers dimensions compress to fit bottlenecks.

:param latent_dim: Core dimension sizing assigned to the target bottleneck space.
:type latent_dim: int
:param user_hyperparameters: Manual overrides configuration maps to bypass automated heuristics.
:type user_hyperparameters: Opti

## Use a preset, then override deliberately

`GradualReduction` derives input width from the active binner and estimates a convolution kernel from sampled peak envelopes. Parameters such as latent and projection dimensions remain explicit. The preset only fills the building buffer, so individual components can still be replaced before compilation. Registered names are preferred over classes or instances because names and parameters are portable JSON.

In [5]:
wrapper.models_manager.set_model_preset(
    "GradualReduction",
    latent_dim=128, # Suggested dimension is much higher, but in this case we are using 16 for faster evaluation
    projection_dim=64,
)
wrapper.models_manager.set_dataset("PixelDataset")
model = wrapper.models_manager.compile_model(run_validation_pass=True)
print(model)

2026-07-19 16:08:01,638 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:108 | Successfully bound active context memory maps for: example
2026-07-19 16:08:01,639 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.architecture_proxy:278 | Initiating model preset configuration layout lookup for family: autoencoder, preset: GradualReduction
2026-07-19 16:08:01,640 | INFO     | msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets.gradual_reduction_preset:46 | Preset builder initiating hyperparameter execution pipeline analysis.
2026-07-19 16:08:08,207 | INFO     | msi_autoencoder_wrapper.models.architectures.utils.presets_utils:88 | Statistical reflection completed. Suggested baseline kernel width: 9 bins
2026-07-19 16:08:08,207 | INFO     | msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets.gradual_reduction_preset:86 | Gradual Reduction layout synthesis finalized. Mapped feature width roadmap

For complete manual construction, call `get_available_components(category)`, then `set_component(category, registered_name, **parameters)` for encoder, decoder, and optional projector/head. Custom components and presets are documented in [Custom models](../../../docs/CUSTOM_MODELS.md).

## Define training phases

Criteria are grouped by where they act. Reconstruction losses consume input and reconstruction; contrastive losses prepare augmented inputs and consume projection outputs; head losses are reserved for named head outputs. A phase may freeze direct child modules by name. Current lifecycle hooks are criterion hooks: phase-start precomputation and batch-start augmentation. No additional inter-layer training hook API is implied here.

In [20]:
training_config = {
    "seed": 1912, # Those who know, know 
    "phases": [
        {
            "phase_name": "joint_reconstruction_and_contrastive",
            "epochs": 3,
            "batch_size": 64,
            "freeze": [],
            "optimizer": {
                "type": "AdamW",
                "params": {"lr": 1e-3, "weight_decay": 1e-4},
            },
            "criterions": {
                "reconstruction": {
                    "mse": {
                        "target": "MSELoss",
                        "weight": 1.0,
                        "params": {},
                    },
                },
                # "contrastive": {
                #     "info_nce": {
                #         "target": "InfoNCELoss",
                #         "weight": 0.05,
                #         "params": {
                #             "temperature": 0.07,
                #             "peak_sample_size": 512,
                #             "peak_sample_seed": 1912,
                #         },
                #     },
                # },
            },
        }
    ],
}

In [17]:
training_config = {
    "seed": 1912, # Those who know, know 
    "phases": [
        {
            "phase_name": "joint_reconstruction_and_contrastive",
            "epochs": 20,
            "batch_size": 64,
            "freeze": [],
            "optimizer": {
                "type": "AdamW",
                "params": {"lr": 1e-3, "weight_decay": 1e-4},
            },
            "criterions": {
                "reconstruction": {
                    "masserstein": {
                        "target": "MassersteinLoss",
                        "weight": 1.0,
                        "params": {},
                    },
                },
                # "contrastive": {
                #     "info_nce": {
                #         "target": "InfoNCELoss",
                #         "weight": 0.05,
                #         "params": {
                #             "temperature": 0.07,
                #             "peak_sample_size": 512,
                #             "peak_sample_seed": 1912,
                #         },
                #     },
                # },
            },
        }
    ],
}

`InfoNCELoss` samples spectra and peak envelopes once per phase, stores a bounded bank in the loaded model's transient training cache, injects sampled envelopes at batch start, and expands an `N` batch to `2N`. Clear the cache explicitly after changing data assumptions with `clear_training_cache()`. Detailed criterion contracts and the Masserstein loss are in [CRITERIONS.md](../../../docs/CRITERIONS.md).

## Estimate capacity before training

The estimator runs one evaluation forward probe and combines observed activation sizes with parameters, gradients, optimizer state, DataLoader buffering, known criterion workspaces, checkpoints, and history. Fractions in `(0, 1]` mean a fraction of currently available resources; larger values mean absolute bytes. It reports RAM, VRAM, and disk separately and can reduce batch size in a copied config. It is an estimate, not a guarantee: native reader caches, allocator fragmentation, OS activity, and future peaks are not fully measurable before training.

In [21]:
#TODO - trzeba deafult printa tutaj zrobić, tak jak przy configurajci, że automatycznei to robi str i może tobie dicty zwrócić, ale może również zrobić tak, że na returnie ustawić print'a
#
report = wrapper.models_manager.estimate_training_resources(
    training_config,
    resource_limits={"ram": 0.65, "vram": 0.80, "disk": 1.00},
    auto_adjust_batch_size=True,
    safety_factor=1.25,
)


for phase in report["phases"]:
    print(
        phase["phase"],
        phase["recommended_batch_size"],
        phase["estimated_ram_bytes"],
        phase["estimated_vram_bytes"],
        phase["fits_limits"],
    )
print(report["estimated_disk_bytes"], report["disk_fits_limit"])
safe_training_config = report["recommended_training_config"]

2026-07-19 16:29:44,718 | INFO     | msi_autoencoder_wrapper.training.resource_estimator:147 | Training resource estimation completed for 1 phase(s).
joint_reconstruction_and_contrastive 64 9347200 756767780 True
886356 True


Masserstein allocates matrices quadratic in the number of m/z bins; InfoNCE allocates similarity matrices quadratic in the expanded batch size. 

For either loss, keep the safety reserve and monitor the first epoch on the target machine.

In [23]:
model_dir = wrapper.workspace.save_model(model_name="tutorial-autoencoder")

2026-07-19 16:31:13,505 | INFO     | msi_autoencoder_wrapper.workspace.model_store:84 | Model configuration saved at: /home/maxi7524/repositories/MSIAutoEncoderWrapper/tutorial_workspace/models/example/tutorial-autoencoder/config/config.json
2026-07-19 16:31:13,519 | INFO     | msi_autoencoder_wrapper.workspace.model_store:157 | Model weights saved at: /home/maxi7524/repositories/MSIAutoEncoderWrapper/tutorial_workspace/models/example/tutorial-autoencoder/config/weights.pt
2026-07-19 16:31:13,521 | INFO     | msi_autoencoder_wrapper.workspace.model_store:210 | Training history saved at: /home/maxi7524/repositories/MSIAutoEncoderWrapper/tutorial_workspace/models/example/tutorial-autoencoder/config/history.json
2026-07-19 16:31:13,522 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.io_models_proxy:194 | Active model 'tutorial-autoencoder' saved under context 'example'.


In [22]:
# Training may be expensive; run after reviewing the resource report.
#TODO - there is problem with not implemented on batch start - each criterion should have default implemention of those, which be deafult do nothing, just ensure compatibility
# mse only 

history = wrapper.models_manager.fit(safe_training_config)
model_dir = wrapper.workspace.save_model(model_name="tutorial-autoencoder")

2026-07-19 16:29:46,214 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.training_proxy:133 | Instantiating execution training manager instance.
2026-07-19 16:29:46,216 | INFO     | msi_autoencoder_wrapper.training.training_manager:78 | Training lifecycle orchestration triggered. Dispatching execution parameters for model family: autoencoder
2026-07-19 16:29:46,217 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:304 | Pre-flight validation successful. Training environment maps verified.
2026-07-19 16:29:46,218 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:63 | Enforcing global deterministic execution pipeline using seed token: 1912
2026-07-19 16:29:46,220 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:89 | Initiating sequential training loop phase: joint_reconstruction_and_contrastive (1/1)
2026-07-19 16:29:46,222 | INFO     | msi_autoencoder_wrapper.training.criterions.criterions_manager:253 | Assembling '

/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)


2026-07-19 16:29:46,482 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 1/3 | Batch 1/545 | Loss: nan | Elapsed: 0.3 s | ETA: 02:16


/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: invalid value encountered in multiply
  return bound(*args, **kwds)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: invalid value encountered in multiply
  return bound(*args, **kwds)


2026-07-19 16:29:49,222 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 1/3 | Batch 110/545 | Loss: nan | Elapsed: 3.0 s | ETA: 00:11
2026-07-19 16:29:51,949 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 1/3 | Batch 219/545 | Loss: nan | Elapsed: 5.7 s | ETA: 00:08
2026-07-19 16:29:54,653 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 1/3 | Batch 328/545 | Loss: nan | Elapsed: 8.4 s | ETA: 00:05
2026-07-19 16:29:57,379 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 1/3 | Batch 437/545 | Loss: nan | Elapsed: 11.1 s | ETA: 00:02
2026-07-19 16:30:00,082 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 1/3 | Batch 545/545 | Loss: nan | Elapsed: 13.

...

2026-07-19 16:30:03,023 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 2/3 | Batch 110/545 | Loss: nan | Elapsed: 2.9 s | ETA: 00:11
2026-07-19 16:30:05,789 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 2/3 | Batch 219/545 | Loss: nan | Elapsed: 5.7 s | ETA: 00:08
2026-07-19 16:30:08,507 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 2/3 | Batch 328/545 | Loss: nan | Elapsed: 8.4 s | ETA: 00:05
2026-07-19 16:30:11,255 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 2/3 | Batch 437/545 | Loss: nan | Elapsed: 11.1 s | ETA: 00:02
2026-07-19 16:30:13,932 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 2/3 | Batch 545/545 | Loss: nan | Elapsed: 13.

/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: invalid value encountered in multiply
  return bound(*args, **kwds)


2026-07-19 16:30:14,119 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 3/3 | Batch 1/545 | Loss: nan | Elapsed: 0.1 s | ETA: 01:20
2026-07-19 16:30:16,803 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 3/3 | Batch 110/545 | Loss: nan | Elapsed: 2.8 s | ETA: 00:11
2026-07-19 16:30:19,550 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 3/3 | Batch 219/545 | Loss: nan | Elapsed: 5.6 s | ETA: 00:08
2026-07-19 16:30:22,274 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 3/3 | Batch 328/545 | Loss: nan | Elapsed: 8.3 s | ETA: 00:05


/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: invalid value encountered in multiply
  return bound(*args, **kwds)


2026-07-19 16:30:25,157 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 3/3 | Batch 437/545 | Loss: nan | Elapsed: 11.2 s | ETA: 00:02
2026-07-19 16:30:27,911 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 3/3 | Batch 545/545 | Loss: nan | Elapsed: 13.9 s | ETA: 00:00
2026-07-19 16:30:27,942 | INFO     | msi_autoencoder_wrapper.workspace.model_store:210 | Training history saved at: /home/maxi7524/repositories/MSIAutoEncoderWrapper/tutorial_workspace/models/example/tutorial-autoencoder/config/history.json
2026-07-19 16:30:27,943 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:238 | === Epoch Summary [joint_reconstruction_and_contrastive] 003/003 | Avg Loss: nan | Patience: 3/10 | Duration: 13.98 s ===
2026-07-19 16:30:27,943 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:253 | All configured sequential multi-phase

In [19]:
# Training may be expensive; run after reviewing the resource report.
#TODO - there is problem with not implemented on batch start - each criterion should have default implemention of those, which be deafult do nothing, just ensure compatibility
# masserstein

history = wrapper.models_manager.fit(safe_training_config)
model_dir = wrapper.workspace.save_model(model_name="tutorial-autoencoder")

2026-07-19 16:26:35,763 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.training_proxy:133 | Instantiating execution training manager instance.
2026-07-19 16:26:35,764 | INFO     | msi_autoencoder_wrapper.training.training_manager:78 | Training lifecycle orchestration triggered. Dispatching execution parameters for model family: autoencoder
2026-07-19 16:26:35,765 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:304 | Pre-flight validation successful. Training environment maps verified.
2026-07-19 16:26:35,766 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:63 | Enforcing global deterministic execution pipeline using seed token: 1912
2026-07-19 16:26:35,769 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:89 | Initiating sequential training loop phase: joint_reconstruction_and_contrastive (1/1)
2026-07-19 16:26:35,770 | INFO     | msi_autoencoder_wrapper.training.criterions.criterions_manager:253 | Assembling '

/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: invalid value encountered in multiply
  return bound(*args, **kwds)


2026-07-19 16:29:05,909 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:191 | [joint_reconstruction_and_contrastive] Epoch 1/20 | Batch 1/545 | Loss: nan | Elapsed: 150.1 s | ETA: 1361:11


KeyboardInterrupt: 